Load Surface Pressure Data

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import torch
from sklearn.preprocessing import MinMaxScaler
from scipy.spatial import cKDTree

# Project paths
project_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "experiments" / "02_climate").is_dir()
)

experiment_dir = project_root / "experiments" / "02_climate"
data_dir = project_root / "datasets" / "climate" / "raw"
output_dir = experiment_dir / "data"
output_dir.mkdir(parents=True, exist_ok=True)

# Load original Surface Pressure NetCDF
pressure_path = next(data_dir.glob("ps_EUR-11_*.nc"))
ds = xr.open_dataset(pressure_path)

# First 400 timesteps, following the author's experiment
pressure = ds["ps"].isel(time=slice(0, 400))

print("Dataset:", pressure_path.name)
print("Variable:", pressure.name)
print("Shape:", pressure.shape)
print("Dimensions:", pressure.dims)
print("Units:", pressure.attrs.get("units"))
print("Time steps:", pressure.sizes["time"])


Dataset: ps_EUR-11_MPI-M-MPI-ESM-LR_rcp26_r1i1p1_CLMcom-BTU-CCLM4-8-17_v1_3hr_202001010000-202012312100.nc
Variable: ps
Shape: (400, 412, 424)
Dimensions: ('time', 'rlat', 'rlon')
Units: Pa
Time steps: 400


Extract Surface Pressure at the Same 5 Wind Locations

In [5]:
from scipy.spatial import cKDTree

wind_path = data_dir / "Klima_Daten_10m_3h_2020_RCP26.nc"
wind_pt_path = output_dir / "wind_nearest_5sites.pt"

# 1. Load previously extracted wind time series
wind_saved = torch.load(
    wind_pt_path, map_location="cpu", weights_only=True
)

wind_raw = np.asarray(wind_saved["wind_raw"])
location_indices = [0, 5000, 10000, 15000, 20000]

# 2. Identify the original grid cells using wind time series
sample_times = [0, 1, 2, 3, 4, 10, 20, 30]

with xr.open_dataset(wind_path) as wind_ds:

    assert np.array_equal(ds["rlat"].values, wind_ds["rlat"].values)
    assert np.array_equal(ds["rlon"].values, wind_ds["rlon"].values)

    wind_snapshots = wind_ds["sfcWind"].isel(
        time=sample_times
    ).values

    n_lat, n_lon = wind_snapshots.shape[1:]

    source_signatures = wind_snapshots.reshape(
        len(sample_times), -1
    ).T

    target_signatures = wind_raw[sample_times, :].T

    tree = cKDTree(source_signatures)
    distances, flat_indices = tree.query(target_signatures)

    if not np.allclose(
        source_signatures[flat_indices],
        target_signatures,
        rtol=1e-5,
        atol=1e-5
    ):
        raise ValueError(
            "Cannot match saved wind locations to original NetCDF grid."
        )

    rows, cols = np.unravel_index(
        flat_indices, (n_lat, n_lon)
    )

# 3. Extract pressure at exactly the same grid cells
site_rows = xr.DataArray(rows, dims="site")
site_cols = xr.DataArray(cols, dims="site")

pressure_raw = (
    pressure.isel(
        rlat=site_rows,
        rlon=site_cols
    )
    .transpose("time", "site")
    .values
    .astype(np.float32)
)

print("Pressure shape:", pressure_raw.shape)

assert pressure_raw.shape == (400, 5)
assert np.isfinite(pressure_raw).all()

# 4. Save dataset for training notebook
save_path = output_dir / "pressure_nearest_5sites.pt"

torch.save({
    "pressure_raw": torch.from_numpy(pressure_raw),
    "location_indices": location_indices,
    "grid_rows": rows.tolist(),
    "grid_cols": cols.tolist(),
    "units": "Pa",
}, save_path)

print("=== SURFACE PRESSURE DATASET ===")
print("Shape:", pressure_raw.shape)
print("Locations:", location_indices)
print("Grid rows:", rows.tolist())
print("Grid cols:", cols.tolist())
print("Pressure range (Pa):",
      round(float(pressure_raw.min()), 2),
      "-",
      round(float(pressure_raw.max()), 2))
print("Saved:", save_path)


Pressure shape: (400, 5)
=== SURFACE PRESSURE DATASET ===
Shape: (400, 5)
Locations: [0, 5000, 10000, 15000, 20000]
Grid rows: [187, 246, 234, 222, 210]
Grid cols: [184, 202, 209, 216, 225]
Pressure range (Pa): 91840.41 - 103480.45
Saved: c:\Users\Win 11 New\Desktop\pa gk\experiments\02_climate\data\pressure_nearest_5sites.pt
